In [1]:
!nvidia-smi  # Verify GPU is available

Mon Dec 29 07:05:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#Mount Google Drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
#Install dependencies

!pip install opencv-python pycocotools huggingface_hub
!pip install torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install decord

!pip uninstall -y transformers
!pip install git+https://github.com/huggingface/transformers.git

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 79.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 73.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 56.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.8/866.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 60.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cusparselt-cu12
    Found existing installation: nvidia-c

In [4]:
# Add repo

!rm -rf dataset-SAM2-segmentation
!git clone -b sam3_image_segmentation https://github.com/hellvetica42/dataset-SAM2-segmentation.git
!ls -la dataset-SAM2-segmentation/

!rm -rf dataset-SAM2-segmentation/sam3/sam3

!git clone https://github.com/facebookresearch/sam3.git dataset-SAM2-segmentation/sam3/sam3




Cloning into 'dataset-SAM2-segmentation'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 69 (delta 28), reused 59 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 28.13 KiB | 5.63 MiB/s, done.
Resolving deltas: 100% (28/28), done.
total 40
drwxr-xr-x 7 root root 4096 Dec 29 07:09 .
drwxr-xr-x 1 root root 4096 Dec 29 07:09 ..
drwxr-xr-x 8 root root 4096 Dec 29 07:09 .git
-rw-r--r-- 1 root root   80 Dec 29 07:09 .gitmodules
-rw-r--r-- 1 root root 2682 Dec 29 07:09 README.md
-rw-r--r-- 1 root root   82 Dec 29 07:09 requirements.txt
drwxr-xr-x 2 root root 4096 Dec 29 07:09 sam2
drwxr-xr-x 3 root root 4096 Dec 29 07:09 sam3
drwxr-xr-x 3 root root 4096 Dec 29 07:09 scripts
drwxr-xr-x 2 root root 4096 Dec 29 07:09 .vscode
Cloning into 'dataset-SAM2-segmentation/sam3/sam3'...
remote: Enumerating objects: 628, done.
remote: Counting objects: 100% (121/121), done.
remote: Com

In [5]:
from huggingface_hub import login
from google.colab import userdata
from huggingface_hub import snapshot_download

HF_TOKEN=userdata.get('HF_TOKEN')
login(HF_TOKEN)

snapshot_download(repo_id="facebook/sam3", local_dir = "/content/Untitled Folder")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

'/content/Untitled Folder'

In [6]:
%cd dataset-SAM2-segmentation/sam3/sam3

!pip install -e .


/content/dataset-SAM2-segmentation/sam3/sam3
Obtaining file:///content/dataset-SAM2-segmentation/sam3/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 114.2 MB/s eta 0:00:00
  Building editable for sam3 (pyproject.toml) ... done
  Created wheel for sam3: filename=sam3-0.1.0-0.editable-py3-none-any.whl size=15392 sha256=82aa3df1443dd0b0432c994514b5bedb75661f86adbd09bd95933b0c4f3ca26c
  Stored in directory: /tmp/pip-ephem-wheel-cache-2iz0difp/wheels/bb/60/46/9e6304570f8ebe88774f9f27cfa3fd05e5ef2e5

In [7]:
import zipfile
import os

# Path to your zip file in Google Drive (UPDATE THIS PATH)
zip_path = '/content/drive/MyDrive/dataset_edge.zip'

# Extract to project directory
print("Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/sam3_project/')

print("Dataset extracted!")
!ls -la /content/sam3_project/

Extracting dataset...
Dataset extracted!
total 12
drwxr-xr-x 3 root root 4096 Dec 29 07:11 .
drwxr-xr-x 1 root root 4096 Dec 29 07:11 ..
drwxr-xr-x 5 root root 4096 Dec 29 07:11 dataset_edge


In [8]:
import os
from pathlib import Path

checkpoint_path = Path('/content/Untitled Folder')
if checkpoint_path.exists():
    print(f"Checkpoint found: {checkpoint_path}")
else:
    print(f"Checkpoint NOT found at: {checkpoint_path}")
    !find /content/sam3_project -name "*.pt"

Checkpoint found: /content/Untitled Folder


In [23]:
with open('/content/dataset-SAM2-segmentation/scripts/run_segmentation.py', 'r') as f:
    content = f.read()



content = content.replace(
    'CKPT_PATH = "/home/ubuntu/dataset-SAM2-segmentation/sam3/checkpoints/dataset-SAM2-segmentation/sam3/checkpoints"',
    'CKPT_PATH = "/content/Untitled Folder"'
)

with open('/content/dataset-SAM2-segmentation/scripts/run_segmentation.py', 'w') as f:
    f.write(content)

print("Updated checkpoint path")




content = content.replace(
    '    images_dir = pick_folder("Select images folder")',
    '    images_dir = "/content/sam3_project/dataset_edge/rubni slucajevi"'
)

with open('/content/dataset-SAM2-segmentation/scripts/run_segmentation.py', 'w') as f:
    f.write(content)
print("Updated image directory path")




content = content.replace(
    '    json_path = pick_json_file("Select JSON annotation file")',
    '    json_path = "/content/sam3_project/dataset_edge/label/instances_Train.json"  '
)

with open('/content/dataset-SAM2-segmentation/scripts/run_segmentation.py', 'w') as f:
    f.write(content)

print("Updated label .json path")




content = content.replace(
    '    out_dir = images_dir.parent / "labels_with_segmentation"',
    '    out_dir = "/content/sam3_project/dataset_edge/labels_with_segmentation"'
)

with open('/content/dataset-SAM2-segmentation/scripts/run_segmentation.py', 'w') as f:
    f.write(content)

print("Updated output directory")




content = content.replace(
    '    out_dir.mkdir(parents=True, exist_ok=True)',
    '    os.mkdir(out_dir)'
)

with open('/content/dataset-SAM2-segmentation/scripts/run_segmentation.py', 'w') as f:
    f.write(content)

print("Create output directory")




content = content.replace(
    '    out_json = out_dir / json_path.name.replace(".json", "_segmented.json")',
    '    out_json = "/content/sam3_project/dataset_edge/label/instances_Train_segmented.json"'
)

with open('/content/dataset-SAM2-segmentation/scripts/run_segmentation.py', 'w') as f:
    f.write(content)

print("Updated output filename")




Updated checkpoint path
Updated image directory path
Updated label .json path
Updated output directory
Create output directory
Updated output filename


In [24]:
#Run your segmentation script
%cd /content/dataset-SAM2-segmentation/scripts

!python3 run_segmentation.py

/content/dataset-SAM2-segmentation/scripts
Select the images folder…
Select the JSON annotation file…

Using:
  Images folder: /content/sam3_project/dataset_edge/rubni slucajevi
  JSON file: /content/sam3_project/dataset_edge/label/instances_Train.json
  Output will be saved to: /content/sam3_project/dataset_edge/label/instances_Train_segmented.json

Initializing SAM3 model...
  Checkpoint: /content/Untitled Folder
  Device: cuda
config.json: 100% 25.8k/25.8k [00:00<00:00, 49.7MB/s]
model.safetensors: 100% 3.44G/3.44G [00:28<00:00, 121MB/s]
Loading weights: 100% 1468/1468 [00:02<00:00, 710.88it/s, Materializing param=vision_encoder.neck.fpn_layers.3.proj2.weight]
processor_config.json: 100% 1.71k/1.71k [00:00<00:00, 7.56MB/s]
tokenizer_config.json: 100% 799/799 [00:00<00:00, 3.69MB/s]
vocab.json: 100% 862k/862k [00:00<00:00, 26.7MB/s]
merges.txt: 100% 525k/525k [00:00<00:00, 2.85MB/s]
tokenizer.json: 100% 3.64M/3.64M [00:00<00:00, 11.5MB/s]
special_tokens_map.json: 100% 588/588 [00:00<

In [ ]:
!ls -lh /content/sam3_project/labels_with_segmentation/


In [ ]:
from google.colab import files

# Download the segmented JSON
files.download('/content/sam3_project/dataset_edge/label/instances_Train_segmented.json')